=============================================================================
SCHEDULE TOOL — REVIEWED & FIXED VERSION
Author review: 2026-05-10
All comments in English. Bugs fixed, edge cases handled, structure improved.
=============================================================================
HOW TO USE:
  Each section is separated by "# %% [CELL N]" — paste into Jupyter as cells,
  or run the whole file as a script.
=============================================================================


## IMPORTS ────────────────────────────────────────────────────


In [1]:
# Core data libraries
import pandas as pd
import numpy as np
import glob
import os
import shutil
import tempfile
from pathlib import Path
from collections import defaultdict, OrderedDict
from datetime import date, datetime, timedelta, time

# Excel I/O
import xlsxwriter                           # write-only Excel engine
import fastexcel                            # fast .xlsx reader (dependency of polars)

# Polars — fast IO and melt/group operations
import polars as pl

# Graph analysis for swap chains
import networkx as nx

# Jupyter display helpers
from IPython.display import display, HTML



## HELPER: input_data() ───────────────────────────────────────


In [2]:
# Reads all .xlsx / .csv files in a folder, melts each file from wide to long,
# and returns a unified Polars DataFrame with columns:
#   Week | IEX ID | Name | LOB | TL | Date | Shift

def input_data(folder_path: str) -> pl.DataFrame:
    """
    Scan a folder for schedule files (.xlsx / .csv), read each one,
    normalise headers, melt to long format, and concatenate.

    Edge cases handled:
    - Empty folder           → returns empty DataFrame with correct schema
    - File missing expected  → skips gracefully (only keeps existing id_vars)
    - Date parse failure     → null (coerced) so rows are kept but Date=null
    - CSV with non-string    → cast all columns to String before melt
    """
    file_paths = (
        glob.glob(os.path.join(folder_path, "*.xlsx"))
        + glob.glob(os.path.join(folder_path, "*.csv"))
    )

    df_list = []

    for file in file_paths:
        source_name = os.path.splitext(os.path.basename(file))[0]

        try:
            if file.endswith(".xlsx"):
                # Read without header first so we can promote row 0 as column names
                raw = pl.read_excel(file, has_header=False)
                if raw.is_empty():
                    continue
                new_header = [str(v).strip() for v in raw.row(0)]
                df = raw.slice(1)
                df.columns = new_header

            elif file.endswith(".csv"):
                df = pl.read_csv(file)
                # Cast every column to String so melt produces a uniform type
                df = df.with_columns([pl.col(c).cast(pl.Utf8) for c in df.columns])

            # Tag with source file name as "Week" identifier
            df = df.with_columns(pl.lit(source_name).alias("Week"))

            # Drop administrative columns that are not needed downstream
            cols_to_drop = [c for c in ["Status", "Sec Status"] if c in df.columns]
            if cols_to_drop:
                df = df.drop(cols_to_drop)

            # Keep only id_vars that actually exist in this file
            id_vars = [c for c in ["Week", "IEX ID", "Name", "LOB", "TL"]
                       if c in df.columns]

            # Melt: every non-id column becomes a Date/Shift row pair
            df = df.melt(id_vars=id_vars, variable_name="Date", value_name="Shift")

            # Parse Date — try ISO format first, then datetime-with-time
            df = df.with_columns(
                pl.coalesce([
                    pl.col("Date").str.strptime(pl.Date, "%Y-%m-%d", strict=False),
                    pl.col("Date").str.strptime(pl.Date, "%Y-%m-%d %H:%M:%S", strict=False),
                ]).alias("Date")
            )

            df_list.append(df)

        except Exception as exc:
            print(f"[WARN] Skipping {file}: {exc}")

    if not df_list:
        # Return an empty frame with the expected schema so callers don't crash
        return pl.DataFrame(schema={
            "Week": pl.Utf8, "IEX ID": pl.Utf8, "Name": pl.Utf8,
            "LOB": pl.Utf8, "TL": pl.Utf8, "Date": pl.Date, "Shift": pl.Utf8,
        })

    return pl.concat(df_list, how="diagonal")   # diagonal tolerates schema diffs



## HELPER: process_schedule_shift() ───────────────────────────


In [3]:
# Splits 'HHMM-HHMM' shift strings into Start_Shift / End_Shift time objects,
# then derives Day_Type (Day/Night) and Shift_Type (Morning/Afternoon/Night).
#
# NOTE: This function operates on a PANDAS DataFrame (after converting from Polars).

def _classify_day_type(shift_time) -> str:
    """Return 'Night_Shift' if start >= 18:00, else 'Day_Shift'."""
    if isinstance(shift_time, time):
        return "Night_Shift" if shift_time >= time(18, 0) else "Day_Shift"
    return np.nan


def _classify_shift_type(shift_time):
    """Bucket shift start into Morning / Afternoon / Night / Other."""
    if not isinstance(shift_time, time):
        return None
    if time(5, 0) <= shift_time < time(12, 0):
        return "Morning"
    if time(12, 0) <= shift_time < time(18, 0):
        return "Afternoon"
    if shift_time >= time(18, 0):
        return "Night"
    return "Other"   # FIX: was 'Day_Shift' — avoids confusion with Day_Type label


def process_schedule_shift(df: pd.DataFrame, shift_col: str) -> pd.DataFrame:
    """
    Parse a 'HHMM-HHMM' column and add Start_Shift, End_Shift,
    Day_Type, and Shift_Type columns.

    Edge cases:
    - Shift values without '-'   → Start/End = NaT, classifications = NaN/None
    - Malformed time strings     → coerced to NaT (errors='coerce')
    """
    mask = df[shift_col].astype(str).str.contains("-", na=False)

    parts = df.loc[mask, shift_col].astype(str).str.split("-", n=1)
    df["Start_Shift"] = np.where(mask, parts.str[0], None)
    df["End_Shift"]   = np.where(mask, parts.str[1], None)

    df["Start_Shift"] = pd.to_datetime(
        df["Start_Shift"], format="%H%M", errors="coerce"
    ).dt.time
    df["End_Shift"] = pd.to_datetime(
        df["End_Shift"], format="%H%M", errors="coerce"
    ).dt.time

    df["Day_Type"]   = df["Start_Shift"].apply(_classify_day_type)
    df["Shift_Type"] = df["Start_Shift"].apply(_classify_shift_type)
    return df



## HELPER: get_most_frequent_shift() ──────────────────────────


In [4]:
# For each agent-week, returns only their most commonly scheduled shift
# (ignores leave codes, keeps only 'HHMM-HHMM' patterns).

def get_most_frequent_shift(df: pl.DataFrame) -> pl.DataFrame:
    """
    Filter to rows where Shift matches 'HHMM-HHMM', group by agent/week,
    count occurrences, and return the top shift per group.

    If NO valid shift exists for an agent, that agent is EXCLUDED from output
    (intentional — a placeholder 'No Shift' row is not useful downstream).
    """
    GROUP_KEYS = ["Week", "IEX ID", "Name", "TL", "LOB"]

    # Keep only columns that actually exist (tolerates missing TL / LOB)
    existing_keys = [c for c in GROUP_KEYS if c in df.columns]
    df = df.select(existing_keys + ["Shift"])

    # Keep only rows with a valid shift pattern (contains '-')
    df_valid = df.filter(pl.col("Shift").str.contains("-"))

    if df_valid.is_empty():
        # Return empty frame — callers must handle this
        return df_valid

    return (
        df_valid
        .group_by(existing_keys + ["Shift"])
        .agg(pl.len().alias("Count"))
        .sort(existing_keys + ["Count"], descending=[False] * len(existing_keys) + [True])
        # Keep only the highest-count shift per agent per week
        .filter(
            pl.col("Count") == pl.col("Count").max().over(existing_keys)
        )
        .drop("Count")
    )



## HELPER: load_schedule_of_schedule() ────────────────────────


In [5]:
# Full pipeline: read folder → most-frequent shift → parse times → return pandas

def load_schedule_of_schedule(folder_path: str) -> pd.DataFrame:
    """
    Read all schedule files in folder_path, keep the dominant shift per agent
    per week, add time-based columns, and return a pandas DataFrame.

    Returns an empty DataFrame if no valid data is found.
    """
    pl_df = input_data(folder_path)

    if pl_df.is_empty():
        print(f"[WARN] No schedule data found in: {folder_path}")
        return pd.DataFrame()

    pl_df = get_most_frequent_shift(pl_df)

    if pl_df.is_empty():
        print(f"[WARN] No valid HHMM-HHMM shifts found in: {folder_path}")
        return pd.DataFrame()

    pd_df = pl_df.to_pandas()
    pd_df = process_schedule_shift(pd_df, "Shift")
    pd_df["IEX ID"] = pd.to_numeric(pd_df["IEX ID"], errors="coerce")
    return pd_df



## HELPER: load_and_merge_hc_data() ───────────────────────────


In [6]:
# Read the VN team headcount workbook (Active + Inactive sheets) and
# return a single normalised pandas DataFrame.

def load_and_merge_hc_data(excel_path: str) -> pd.DataFrame:
    """
    Merge Active and Inactive HC sheets.
    Missing columns are filled with pd.NA so the schema is always consistent.
    """
    SELECTED_COLS = [
        "Site", "IEX ID", "OracleID", "Employee Name",
        "Alias", "Gender", "Email Id", "LWD/Movement", "Mini Team",
    ]

    def _read_sheet(sheet: str) -> pd.DataFrame:
        raw = pd.read_excel(excel_path, sheet_name=sheet, header=None)
        raw.columns = raw.iloc[0]
        raw = raw.iloc[1:].reset_index(drop=True)
        # Add any missing columns with NA so concat doesn't fail
        for col in SELECTED_COLS:
            if col not in raw.columns:
                raw[col] = pd.NA
        return raw[SELECTED_COLS]

    combined = pd.concat([_read_sheet("Active"), _read_sheet("Inactive")],
                         ignore_index=True)
    combined["LWD/Movement"] = pd.to_datetime(combined["LWD/Movement"], errors="coerce")
    combined["IEX ID"] = pd.to_numeric(combined["IEX ID"], errors="coerce")
    return combined



## CONFIG ──────────────────────────────────────────────────────


In [7]:
# ==============================================================================
# ⚙️  CONFIG — Edit this cell every week before running the notebook
# ==============================================================================

HOME = os.path.expanduser("~").replace("\\", "/")
BASE = f"{HOME}/Concentrix Corporation/WFM-Expedia-HCM - Branding files"

# Validate base directory
if not os.path.exists(BASE):
    raise FileNotFoundError(
        f"Base directory not found: {BASE}\n"
        "Please ensure OneDrive is synced."
    )

# ------------------------------------------------------------------------------
# ✏️  CHANGE THESE TWO VALUES EVERY WEEK
# ------------------------------------------------------------------------------
WEEK        = "Schedule_2026_05_18"   # Format: Schedule_YYYY_MM_DD (Monday)
VN_TEAM_FILE = (                       # ✏️ Update filename each week
    f"{BASE}/Schedule/Schedule (RTA version)/2026/Schedule_WB0518.xlsx"
)
# Pattern: Schedule_WB{MMDD}.xlsx  e.g. week 18-May → Schedule_WB0518.xlsx

# ------------------------------------------------------------------------------
# Folder paths (derived automatically — no need to edit below)
# ------------------------------------------------------------------------------
FOLDER_PATHS = {
    "schedule_team":  f"{BASE}/Rawdata/INPUT_SCHEDULE/SCHEDULE_TEAM",
    "schedule_vn":    f"{BASE}/Rawdata/INPUT_SCHEDULE/SCHEDULE_VN",
    "analysis":       f"{BASE}/Rawdata/INPUT_SCHEDULE/SCHEDULE_ANALYSIS",
    "schedule_step1": f"{BASE}/Rawdata/INPUT_SCHEDULE/SCHEDULE_STEP_1",
    "vn_team":        VN_TEAM_FILE,
    "hc_master":     f"{BASE}/Headcount/HC Master Database - 2026.xlsx"
}

# Create output directories if they don't exist
for key in ["schedule_step1", "analysis"]:
    os.makedirs(FOLDER_PATHS[key], exist_ok=True)

# ------------------------------------------------------------------------------
# ✅  Validate all paths before continuing — fail fast with a clear message
# ------------------------------------------------------------------------------
print("[CONFIG] Validating paths...")
errors = []
for name, path in FOLDER_PATHS.items():
    exists = os.path.exists(path)
    status = "✅" if exists else "❌"
    print(f"  {status} {name}: {path}")
    if not exists:
        errors.append(f"{name}: {path}")

if errors:
    raise FileNotFoundError(
        "The following paths were NOT found — please update CONFIG before continuing:\n"
        + "\n".join(errors)
    )

print(f"\n[CONFIG] ✅ All paths OK")
print(f"[CONFIG] Week     : {WEEK}")
print(f"[CONFIG] VN Team  : {VN_TEAM_FILE}")


[CONFIG] Validating paths...
  ✅ schedule_team: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/INPUT_SCHEDULE/SCHEDULE_TEAM
  ✅ schedule_vn: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/INPUT_SCHEDULE/SCHEDULE_VN
  ✅ analysis: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/INPUT_SCHEDULE/SCHEDULE_ANALYSIS
  ✅ schedule_step1: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/INPUT_SCHEDULE/SCHEDULE_STEP_1
  ✅ vn_team: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Schedule/Schedule (RTA version)/2026/Schedule_WB0518.xlsx
  ✅ hc_master: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Master Database - 2026.xlsx

[CONFIG] ✅ All paths OK
[CONFIG] Week     : Schedule_2026_05_18
[CONFIG] VN Team  : C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Schedule/Schedule (RTA version)/2026/Sch

## LOAD & COMBINE SCHEDULES ───────────────────────────────────


In [8]:
# Load both Schedule_Team and Schedule_VN sources, label them, concatenate,
# and export per-week analysis snapshots.

Schedule_Of_Schedule_Team = load_schedule_of_schedule(FOLDER_PATHS["schedule_team"])
Schedule_Of_Schedule_Team["Source"] = "Schedule_Team"

Schedule_Of_Schedule_VN = load_schedule_of_schedule(FOLDER_PATHS["schedule_vn"])
Schedule_Of_Schedule_VN["Source"] = "VN_Team"

Combine_Schedule = pd.concat(
    [Schedule_Of_Schedule_Team, Schedule_Of_Schedule_VN], ignore_index=True
)

# Export one Excel file per week into the analysis folder
for week_val, group in Combine_Schedule.groupby("Week"):
    out = os.path.join(FOLDER_PATHS["analysis"], f"{week_val}.xlsx")
    group.to_excel(out, sheet_name=str(week_val)[:31], index=False)   # sheet name ≤31 chars

print(f"[DONE] Combined schedule: {len(Combine_Schedule)} rows across "
      f"{Combine_Schedule['Week'].nunique()} weeks.")



C:\Users\ADMIN\AppData\Local\Temp\ipykernel_19632\1877741548.py:54: DeprecationWarning: `DataFrame.melt` is deprecated; use `DataFrame.unpivot` instead, with `index` instead of `id_vars` and `on` instead of `value_vars`
  df = df.melt(id_vars=id_vars, variable_name="Date", value_name="Shift")
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_19632\1877741548.py:54: DeprecationWarning: `DataFrame.melt` is deprecated; use `DataFrame.unpivot` instead, with `index` instead of `id_vars` and `on` instead of `value_vars`
  df = df.melt(id_vars=id_vars, variable_name="Date", value_name="Shift")


[DONE] Combined schedule: 16989 rows across 99 weeks.


## STEP 8: LOAD SCHEDULE, AUDIT PLANNED LEAVE, RESTORE & SWAP ─


In [9]:
# This is the main processing cell.  Steps:
#   1. Load the global schedule for the target week
#   2. Load and audit Planned Leave vs global schedule
#   3. Restore dominant (base) shift where global leave codes were placed
#   4. Load Swap data and apply swaps / self-changes
#   5. Build NetworkX graph and attach Detailed Swap History + Swap Steps

# ── 1. LOAD GLOBAL SCHEDULE ──────────────────────────────────────────────────
schedule_step_1 = pd.read_excel(
    os.path.join(FOLDER_PATHS["schedule_team"], f"{WEEK}.xlsx")
)
schedule_step_1["Week"] = WEEK

# Ensure 'Week' is the first column
cols = ["Week"] + [c for c in schedule_step_1.columns if c != "Week"]
Schedule_VN = schedule_step_1[cols].copy()
Schedule_VN["Planned Note"] = ""

# Date columns: any column whose string representation starts with '202'
DATE_COLS = [str(c) for c in Schedule_VN.columns if str(c).startswith("202")]

# ── 2. LOAD PLANNED LEAVE ────────────────────────────────────────────────────
# Use a temp copy to avoid locking the OneDrive file
def _safe_read_excel(source_path: str, **kwargs) -> pd.DataFrame:
    """Copy file to temp location, read, then delete — avoids file-lock issues."""
    tmp = os.path.join(tempfile.gettempdir(), f"_tmp_{os.path.basename(source_path)}")
    shutil.copy2(source_path, tmp)
    try:
        return pd.read_excel(tmp, **kwargs)
    finally:
        try:
            os.remove(tmp)
        except OSError:
            pass


raw_planned = _safe_read_excel(FOLDER_PATHS["vn_team"], sheet_name="Planned", header=None)
raw_planned.columns = raw_planned.iloc[0]
Schedule_Planned = raw_planned.iloc[1:].reset_index(drop=True)

PLANNED_COLS = ["Leave Date", "Week", "Type of Request", "IEX ID", "Agent Name", "Planned"]
# Keep only columns that exist (defensive)
Schedule_Planned = Schedule_Planned[[c for c in PLANNED_COLS if c in Schedule_Planned.columns]]
Schedule_Planned["Leave Date"] = pd.to_datetime(
    Schedule_Planned["Leave Date"], errors="coerce"
).dt.strftime("%Y-%m-%d")
Schedule_Planned["IEX ID"] = pd.to_numeric(Schedule_Planned["IEX ID"], errors="coerce")

# ── 3. AUDIT: GLOBAL vs LOCAL PLANNED LEAVE ──────────────────────────────────
print("\n🔍 CROSS-CHECKING PLANNED LEAVE...")

LEAVE_CODES_GLOBAL = {"PTO", "AL", "PAID LEAVE", "CO", "HOLIDAY", "HO", "LWP"}

# Collect all leave entries found in the global schedule
global_leaves = []
for _, row in Schedule_VN.iterrows():
    iex = pd.to_numeric(row.get("IEX ID"), errors="coerce")
    for d in DATE_COLS:
        val = str(row.get(d, "")).strip().upper()
        if val in LEAVE_CODES_GLOBAL:
            global_leaves.append({"IEX ID": iex, "Name": row.get("Name", ""), "Date": d, "Global_Code": val})

df_global_leaves = pd.DataFrame(global_leaves) if global_leaves else \
    pd.DataFrame(columns=["IEX ID", "Name", "Date", "Global_Code"])

df_local_leaves = (
    Schedule_Planned[["IEX ID", "Agent Name", "Leave Date", "Type of Request"]]
    .rename(columns={"Leave Date": "Date", "Type of Request": "Local_Code", "Agent Name": "Name_local"})
    .copy()
)
# Only audit dates that appear in the current week's schedule
df_local_leaves = df_local_leaves[df_local_leaves["Date"].isin(DATE_COLS)]

import re

def _normalise_name(raw) -> str:
    """Normalise 'Last, CamelCaseFirst' → 'FIRST LAST' uppercase. Pass-through if no comma."""
    s = str(raw).strip()
    if not s or s.lower() in ('nan', 'none', ''):
        return ''
    if ',' in s:
        parts = [p.strip() for p in s.split(',', 1)]
        first = re.sub(r'(?<=[a-z])(?=[A-Z])', ' ', parts[1])  # camelCase → words
        s = first + ' ' + parts[0]
    return s.upper().strip()

NORM_MAP = {"PTO": "AL", "AL": "AL", "PAID LEAVE": "CO", "CO": "CO",
            "HOLIDAY": "HO", "HO": "HO", "LWP": "LWP"}

merged_leaves = pd.merge(df_local_leaves, df_global_leaves, on=["IEX ID", "Date"], how="outer")
discrepancies = []

for _, row in merged_leaves.iterrows():
    iex, dt = row["IEX ID"], row["Date"]
    l_raw = str(row.get("Local_Code", "")).strip().upper()
    g_raw = str(row.get("Global_Code", "")).strip().upper()
    local_empty  = l_raw in ("", "NAN", "NONE") or pd.isna(row.get("Local_Code"))
    global_empty = g_raw in ("", "NAN", "NONE") or pd.isna(row.get("Global_Code"))
    # Name: prefer local Planned name (clean), fall back to global schedule name (normalised)
    name = _normalise_name(
        row['Name_local'] if pd.notna(row.get('Name_local')) else row.get('Name', '')
    )

    if local_empty and not global_empty:
        discrepancies.append({"IEX ID": iex, "Name": name, "Date": dt,
            "Local Request": "Not in list", "Global Schedule": g_raw,
            "Issue": "Global has Leave not in Planned list"})
    elif not local_empty and global_empty:
        actual = "No data"
        if dt in Schedule_VN.columns:
            m = Schedule_VN["IEX ID"] == iex
            if m.any():
                actual = Schedule_VN.loc[m, dt].iloc[0]
        discrepancies.append({"IEX ID": iex, "Name": name, "Date": dt,
            "Local Request": l_raw, "Global Schedule": actual,
            "Issue": "Global missed / Scheduled over leave"})
    elif not local_empty and not global_empty:
        if NORM_MAP.get(l_raw, l_raw) != NORM_MAP.get(g_raw, g_raw):
            discrepancies.append({"IEX ID": iex, "Name": name, "Date": dt,
                "Local Request": l_raw, "Global Schedule": g_raw,
                "Issue": "Mismatched Leave Code"})

df_discrepancies = pd.DataFrame(discrepancies)
if not df_discrepancies.empty:
    print(f"⚠️  {len(df_discrepancies)} discrepancies found:")
    display(df_discrepancies)
else:
    print("✅ Global schedule perfectly matches the Local Planned Leave list.")
print("=" * 80)

# ── 4. RESTORE DOMINANT SHIFT ────────────────────────────────────────────────
LEAVE_CODES_RESTORE = {"PTO", "PAID LEAVE", "HOLIDAY", "AL", "CO", "HO", "LWP", "LEAVE"}

def restore_dominant_shift(df: pd.DataFrame, date_cols: list) -> pd.DataFrame:
    """
    For each agent row, compute the most-common HHMM-HHMM shift across the week.
    Replace any global leave-code cells with that dominant shift.

    Edge cases:
    - Agent has NO valid shift this week → dominant = '0000-0000' (sentinel)
    - All days are leave codes           → all replaced with '0000-0000'
    """
    print("🔍 Restoring base schedule (replacing Global leave codes)...")
    log = []

    for idx, row in df.iterrows():
        shifts_only = [str(row[d]).strip() for d in date_cols if "-" in str(row[d])]
        mode_result = pd.Series(shifts_only).mode()
        dominant = mode_result.iloc[0] if not mode_result.empty else "0000-0000"

        for d in date_cols:
            cell = str(row[d]).strip().upper()
            if cell in LEAVE_CODES_RESTORE:
                log.append({"IEX ID": row.get("IEX ID"), "Name": row.get("Name"),
                             "Date": d, "Was": cell, "Restored_To": dominant})
                df.at[idx, d] = dominant

    df_log = pd.DataFrame(log)
    if not df_log.empty:
        print(f"✅ Restored {len(df_log)} leave-code cells to dominant shifts.")
        display(df_log.style.hide(axis="index").set_properties(**{"text-align": "center"}))
    else:
        print("✅ No global leave codes to restore this week.")
    return df


Schedule_VN = restore_dominant_shift(Schedule_VN, DATE_COLS)

# ── 5. LOAD SWAP DATA ────────────────────────────────────────────────────────
raw_swap = _safe_read_excel(FOLDER_PATHS["vn_team"], sheet_name="Swap", header=None)
raw_swap.columns = raw_swap.iloc[0]
Schedule_Swap = raw_swap.iloc[1:].reset_index(drop=True)

# Filter to current week only
Schedule_Swap = Schedule_Swap[Schedule_Swap["Week"] == WEEK].copy()

for c in ["IEX1", "IEX2"]:
    if c in Schedule_Swap.columns:
        Schedule_Swap[c] = pd.to_numeric(Schedule_Swap[c], errors="coerce")

SWAP_COLS = ["Week", "IEX1", "IEX2", "Name1", "Name2", "Step",
             "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
# Keep only columns that exist
Schedule_Swap = Schedule_Swap[[c for c in SWAP_COLS if c in Schedule_Swap.columns]]
Schedule_Swap = Schedule_Swap.reset_index(drop=True)
Schedule_Swap["_row_order_"] = Schedule_Swap.index

# Build week date mapping (Monday = WEEK date)
WEEK_START = datetime.strptime(WEEK.replace("Schedule_", ""), "%Y_%m_%d")
WEEKDAYS   = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
DAY_TO_DATE = {d: (WEEK_START + timedelta(days=i)).strftime("%Y-%m-%d")
               for i, d in enumerate(WEEKDAYS)}
DAY_ABBR    = {d: d[:3] for d in WEEKDAYS}

Schedule_VN["IEX ID"] = pd.to_numeric(Schedule_VN["IEX ID"], errors="coerce")

# ── 6. APPLY SWAPS ───────────────────────────────────────────────────────────
def _swap_cells(df: pd.DataFrame, iex1, iex2, date_col: str) -> pd.DataFrame:
    """Swap the value in date_col between two agents. Warns if either not found."""
    mask1 = df["IEX ID"] == iex1
    mask2 = df["IEX ID"] == iex2
    if not mask1.any():
        print(f"  [WARN] IEX {iex1} not found — swap skipped for {date_col}")
        return df
    if not mask2.any():
        print(f"  [WARN] IEX {iex2} not found — swap skipped for {date_col}")
        return df
    idx1, idx2 = df.index[mask1][0], df.index[mask2][0]
    df.at[idx1, date_col], df.at[idx2, date_col] = \
        df.at[idx2, date_col], df.at[idx1, date_col]
    print(f"  [SWAP] {date_col}: {iex1} ↔ {iex2}")
    return df


for _, row in Schedule_Swap.sort_values("_row_order_").iterrows():
    iex1, iex2 = row["IEX1"], row["IEX2"]
    is_self = pd.isna(iex2) or iex1 == iex2

    for day in WEEKDAYS:
        flag = str(row.get(day, "")).strip()
        if pd.isna(row.get(day)) or flag == "":
            continue
        flag_up = flag.upper()
        date_col = DAY_TO_DATE.get(day)
        if not date_col or date_col not in Schedule_VN.columns:
            continue

        if is_self:
            # Self-change: directly write the new shift value
            if flag_up not in {"KEEP", "SWAP", "NAN", "NONE", ""}:
                Schedule_VN.loc[Schedule_VN["IEX ID"] == iex1, date_col] = flag
                print(f"  [MOVE] {day} ({date_col}): {iex1} -> '{flag}'")
        else:
            if flag_up == "SWAP":
                Schedule_VN = _swap_cells(Schedule_VN, iex1, iex2, date_col)

# ── 7. BUILD NETWORKX GRAPH & ATTACH SWAP HISTORY ───────────────────────────
G = nx.Graph()
all_agents: set[int] = set()

for _, row in Schedule_Swap.iterrows():
    iex1, iex2 = row["IEX1"], row["IEX2"]
    if pd.notna(iex1):
        all_agents.add(int(iex1))
        G.add_node(int(iex1))
    if pd.notna(iex2):
        all_agents.add(int(iex2))
        G.add_node(int(iex2))
    if pd.notna(iex1) and pd.notna(iex2) and iex1 != iex2:
        G.add_edge(int(iex1), int(iex2))

components    = list(nx.connected_components(G))
agent_to_comp = {int(a): i for i, comp in enumerate(components) for a in comp}

comp_lines: dict = defaultdict(list)
comp_seen:  dict = defaultdict(set)
iex_steps:  dict = defaultdict(set)

for _, row in Schedule_Swap.sort_values("_row_order_").iterrows():
    iex1, iex2 = row["IEX1"], row["IEX2"]
    name1 = str(row.get("Name1", "") or "")
    name2 = str(row.get("Name2", "") or "")
    step  = str(row.get("Step", "") or "").strip()

    if pd.notna(iex1) and step:
        iex_steps[int(iex1)].add(step)
    if pd.notna(iex2) and step:
        iex_steps[int(iex2)].add(step)

    swap_days = [d for d in WEEKDAYS if str(row.get(d, "")).strip().upper() == "SWAP"]
    day_str = (
        "" if len(swap_days) in (0, 7)
        else " (Swap " + ", ".join(DAY_ABBR[d] for d in swap_days) + ")"
    )

    comp_id = agent_to_comp.get(int(iex1) if pd.notna(iex1) else -1)
    if comp_id is None:
        continue

    if pd.isna(iex2) or iex1 == iex2:
        # Self-change entry
        line = f"{int(iex1)} - {name1} moved to new shift (Step {step})"
    else:
        line = f"{int(iex1)} - {name1} <-> {int(iex2)} - {name2}{day_str} (Step {step})"

    if line not in comp_seen[comp_id]:
        comp_lines[comp_id].append(line)
        comp_seen[comp_id].add(line)

# Attach history and steps back to Schedule_VN
detailed_by_agent = {
    int(a): "\n".join(comp_lines[agent_to_comp[int(a)]])
    for a in all_agents if int(a) in agent_to_comp
}
steps_by_agent = {
    int(a): ", ".join(sorted(iex_steps.get(int(a), set()), key=lambda x: int(x) if x.isdigit() else 0))
    for a in all_agents
}

Schedule_VN["Detailed Swap History"] = (
    Schedule_VN["IEX ID"]
    .map(lambda x: detailed_by_agent.get(int(x), "") if pd.notna(x) else "")
    .fillna("")
)
Schedule_VN["Swap Steps"] = (
    Schedule_VN["IEX ID"]
    .map(lambda x: steps_by_agent.get(int(x), "") if pd.notna(x) else "")
    .fillna("")
)

print("\n[DONE] Swap processing complete.")




🔍 CROSS-CHECKING PLANNED LEAVE...
⚠️  7 discrepancies found:


,IEX ID,Name,Date,Local Request,Global Schedule,Issue
0,3005527,PHAM THANH VU,2026-05-18,CO,0600-1500,Global missed / Scheduled over leave
1,3014817,NGO VAN MINH DUY,2026-05-18,CO,0600-1500,Global missed / Scheduled over leave
2,3014817,NGO VAN MINH DUY,2026-05-19,CO,0600-1500,Global missed / Scheduled over leave
3,3014817,NGO VAN MINH DUY,2026-05-20,CO,0600-1500,Global missed / Scheduled over leave
4,3026476,NGUYEN KHAC BAO THAI,2026-05-23,CO,0600-1500,Global missed / Scheduled over leave
5,3089155,VUU TIEN NHAN,2026-05-18,CO,0600-1500,Global missed / Scheduled over leave
6,3089155,VUU TIEN NHAN,2026-05-21,CO,0600-1500,Global missed / Scheduled over leave


🔍 Restoring base schedule (replacing Global leave codes)...
✅ Restored 12 leave-code cells to dominant shifts.


IEX ID,Name,Date,Was,Restored_To
3092114,"Phan, DatLoi",2026-05-23,PAID LEAVE,0600-1500
3092114,"Phan, DatLoi",2026-05-24,PAID LEAVE,0600-1500
3026455,"Nguyen, TranAnhThu",2026-05-24,PAID LEAVE,0700-1600
3026472,"Ngo, HaTrang",2026-05-19,PAID LEAVE,0600-1500
3097284,"Nguyen, MinhNhat",2026-05-18,PAID LEAVE,2000-0500
3002199,"Le, HoangMaiThy",2026-05-24,PTO,0600-1500
3002196,"Phung, MyUyen",2026-05-21,PTO,2000-0500
3109394,"Bui, NgocThuanVy",2026-05-23,PAID LEAVE,2000-0500
3109394,"Bui, NgocThuanVy",2026-05-24,PAID LEAVE,2000-0500
3090069,Nguyen Thanh Hao,2026-05-18,PAID LEAVE,2200-0700


  [MOVE] Monday (2026-05-18): 3052123 -> 'Flex_Trainer'
  [MOVE] Tuesday (2026-05-19): 3052123 -> 'Flex_Trainer'
  [MOVE] Wednesday (2026-05-20): 3052123 -> 'Flex_Trainer'
  [MOVE] Thursday (2026-05-21): 3052123 -> 'Flex_Trainer'
  [MOVE] Friday (2026-05-22): 3052123 -> 'Flex_Trainer'
  [MOVE] Saturday (2026-05-23): 3052123 -> 'Flex_Trainer'
  [MOVE] Sunday (2026-05-24): 3052123 -> 'Flex_Trainer'
  [SWAP] 2026-05-18: 3089724 ↔ 3090805
  [SWAP] 2026-05-19: 3089724 ↔ 3090805
  [SWAP] 2026-05-20: 3089724 ↔ 3090805
  [SWAP] 2026-05-21: 3089724 ↔ 3090805
  [SWAP] 2026-05-22: 3089724 ↔ 3090805
  [SWAP] 2026-05-23: 3089724 ↔ 3090805
  [SWAP] 2026-05-24: 3089724 ↔ 3090805
  [MOVE] Monday (2026-05-18): 3089724 -> '0700-1600'
  [MOVE] Tuesday (2026-05-19): 3089724 -> '0700-1600'
  [MOVE] Wednesday (2026-05-20): 3089724 -> '0700-1600'
  [MOVE] Thursday (2026-05-21): 3089724 -> 'OFF'
  [MOVE] Friday (2026-05-22): 3089724 -> 'OFF'
  [MOVE] Saturday (2026-05-23): 3089724 -> '1100-2000'
  [MOVE] Sund

## STEP 12: PLOT PLANNED LEAVE & EXPORT STEP-1 ───────────────


In [10]:
# Apply approved leave codes over the swapped schedule, normalise codes,
# count planned days per agent, and export the final Step-1 file.

print("=" * 80)
print("Plotting Planned Leave onto the final swapped schedule...")

for _, row in Schedule_Planned.iterrows():
    col   = str(row.get("Leave Date", ""))
    iex   = row.get("IEX ID")
    req   = str(row.get("Type of Request", "")).strip()
    name  = str(row.get("Agent Name", ""))

    if col not in Schedule_VN.columns:
        continue

    mask = Schedule_VN["IEX ID"] == iex
    if not mask.any():
        continue

    current_val = str(Schedule_VN.loc[mask, col].iloc[0]).strip().upper()

    if current_val != "OFF":
        # Agent is scheduled — apply leave code
        Schedule_VN.loc[mask, col]      = req
        Schedule_VN.loc[mask, "Planned"] = "RTA"
    else:
        # Agent already on OFF — log the conflict
        existing_note = str(Schedule_VN.loc[mask, "Planned Note"].iloc[0])
        new_note = f"{iex} - {name} has {req} on OFF date {col}"
        Schedule_VN.loc[mask, "Planned Note"] = (
            new_note if (pd.isna(existing_note) or existing_note.strip() == "")
            else existing_note + " | " + new_note
        )

# Normalise leave codes to standard abbreviations
CODE_MAP = {
    "PTO":        "AL",
    "PAID LEAVE": "CO",
    "Paid Leave": "CO",
    "HOLIDAY":    "HO",
    "Holiday":    "HO",
}
Schedule_VN[DATE_COLS] = Schedule_VN[DATE_COLS].replace(CODE_MAP)

# Count planned leave days per agent (AL / CO / HO)
Schedule_VN["Planned"] = Schedule_VN[DATE_COLS].apply(
    lambda r: r.astype(str).str.contains(r"^(AL|CO|HO)$", na=False).sum(),
    axis=1,
)
Schedule_VN["Week"] = WEEK

# Export
step1_path = os.path.join(FOLDER_PATHS["schedule_step1"], f"{WEEK}.xlsx")
Schedule_VN.to_excel(step1_path, index=False)
print(f"[✅ COMPLETED] Step-1 file saved: {step1_path}")



Plotting Planned Leave onto the final swapped schedule...
[✅ COMPLETED] Step-1 file saved: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/INPUT_SCHEDULE/SCHEDULE_STEP_1\Schedule_2026_05_18.xlsx


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_19632\2908979002.py:47: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  lambda r: r.astype(str).str.contains(r"^(AL|CO|HO)$", na=False).sum(),
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_19632\2908979002.py:47: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  lambda r: r.astype(str).str.contains(r"^(AL|CO|HO)$", na=False).sum(),
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_19632\2908979002.py:47: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  lambda r: r.astype(str).str.contains(r"^(AL|CO|HO)$", na=False).sum(),
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_19632\2908979002.py:47: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the gr

## HELPER: process_historical_swap_v3() ──────────────────────


In [11]:
# Read a 'Step_2' sheet containing paired rows (agent1 / agent2 with Old/New days),
# detect which days were actually swapped, and return a structured DataFrame.

def process_historical_swap_v3(file_path: str, sheet_name: str = "Step_2") -> pd.DataFrame:
    """
    Parse historical swap data stored as paired rows (agent1 on row i, agent2 on row i+1).
    Detects full swap (all days) vs partial swap and marks each day as SWAP / KEEP.

    Edge cases:
    - Odd number of rows        → last row is ignored with a warning
    - New/Old values are NaN    → treated as empty string (no swap)
    - Both agents identical     → handled but unusual
    """
    DAYS_SHORT = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    DAYS_FULL  = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

    df = _safe_read_excel(file_path, sheet_name=sheet_name)
    results = []

    if len(df) % 2 != 0:
        print(f"[WARN] process_historical_swap_v3: odd row count ({len(df)}) — last row skipped.")

    for i in range(0, len(df) - 1, 2):
        a1, a2 = df.iloc[i], df.iloc[i + 1]
        row = {
            "IEX1": a1.get("IEX"),  "IEX2": a2.get("IEX"),
            "Name1": a1.get("Agent Name"), "Name2": a2.get("Agent Name"),
            "Step": 2,
        }

        old1 = [str(a1.get(f"Old {d}", "")).strip().upper() for d in DAYS_SHORT]
        new1 = [str(a1.get(f"New {d}", "")).strip().upper() for d in DAYS_SHORT]
        old2 = [str(a2.get(f"Old {d}", "")).strip().upper() for d in DAYS_SHORT]
        new2 = [str(a2.get(f"New {d}", "")).strip().upper() for d in DAYS_SHORT]

        # Full-swap: agent1 got agent2's entire schedule and vice versa
        is_full_swap = (new1 == old2) and (new2 == old1)
        swapped = []

        for j, (short, full) in enumerate(zip(DAYS_SHORT, DAYS_FULL)):
            if is_full_swap or old1[j] != new1[j] or old2[j] != new2[j]:
                row[full] = "SWAP"
                swapped.append(short)
            else:
                row[full] = "KEEP"

        row["Swap Days"] = (
            "Full" if (is_full_swap or len(swapped) == 7)
            else (", ".join(swapped) if swapped else "None")
        )
        results.append(row)

    return pd.DataFrame(results)


# Example call (output_path commented out — uncomment to save)
df_historical = process_historical_swap_v3(FOLDER_PATHS["vn_team"], sheet_name="Step_2")
# df_historical.to_excel("<output_path>", index=False)
print(f"[DONE] Historical swap: {len(df_historical)} swap pairs processed.")
display(df_historical)

[DONE] Historical swap: 13 swap pairs processed.


,IEX1,IEX2,Name1,Name2,Step,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,Sunday,Swap Days
0,3085338.0,3026462.0,PHUNG NGOC TRAM,VO TIEN DAT,2,KEEP,SWAP,KEEP,SWAP,SWAP,SWAP,SWAP,"Tue, Thu, Fri, Sat, Sun"
1,3086394.0,3086429.0,NGUYEN HA TU UYEN,NGUYEN THANH TRUNG,2,SWAP,SWAP,SWAP,SWAP,SWAP,SWAP,SWAP,Full
2,3110171.0,3109643.0,NGUYEN NGOC HOANG YEN,TRAN DUC TIN,2,SWAP,SWAP,SWAP,SWAP,SWAP,SWAP,SWAP,Full
3,3026455.0,3085158.0,NGUYEN TRAN ANH THU,TA KHANH HOA,2,SWAP,SWAP,KEEP,SWAP,SWAP,KEEP,KEEP,"Mon, Tue, Thu, Fri"
4,3052315.0,3092314.0,PHUNG HA TRAN,HUY NGUYEN NHAT ANH,2,SWAP,SWAP,SWAP,SWAP,SWAP,SWAP,SWAP,Full
5,3096698.0,3026445.0,NGUYEN NGOC DIEM QUYNH,HO QUY MINH,2,SWAP,SWAP,SWAP,SWAP,SWAP,SWAP,SWAP,Full
6,3026476.0,3026480.0,NGUYEN KHAC BAO THAI,NGO TAN PHAT,2,KEEP,SWAP,SWAP,SWAP,SWAP,SWAP,KEEP,"Tue, Wed, Thu, Fri, Sat"
7,3085252.0,3109399.0,HOANG LE VY,NGUYEN THI KIM NGAN,2,KEEP,KEEP,SWAP,KEEP,SWAP,KEEP,KEEP,"Wed, Fri"
8,3096723.0,3026444.0,MS. HO NGOC HUYEN TRANG,NGUYEN THI HAI LINH,2,SWAP,SWAP,SWAP,SWAP,SWAP,SWAP,SWAP,Full
9,3087487.0,3090069.0,NGUYEN THI TUONG VY,NGUYEN THANH HAO,2,KEEP,KEEP,KEEP,SWAP,SWAP,SWAP,SWAP,"Thu, Fri, Sat, Sun"


## HELPER: get_cell_style() ───────────────────────────────────


In [12]:
# Returns an inline CSS string for Excel/pandas Styler cell colouring
# based on shift value (OFF, leave codes, shift start time).

def get_cell_style(val) -> str:
    """
    Colour-code a schedule cell:
      OFF         → grey
      AL          → green
      CO          → blue
      LWP         → orange
      TERMINATION → red / white text
      HO          → yellow
      Morning     → light blue  (05:00–11:59)
      Afternoon   → light yellow (12:00–17:59)
      Night       → purple      (18:00+)
    """
    if pd.isna(val) or str(val).strip() == "":
        return ""

    v = str(val).strip().upper()

    STATIC_STYLES = {
        "OFF":         "background-color:#A6A6A6;color:black;",
        "AL":          "background-color:#A9D08E;color:black;",
        "CO":          "background-color:#8EA9DB;color:black;",
        "LWP":         "background-color:#F4B084;color:black;",
        "HO":          "background-color:#FFD966;color:black;",
    }
    if v in STATIC_STYLES:
        return STATIC_STYLES[v]
    if "TERMINATION" in v:
        return "background-color:#FF0000;color:white;"

    # Shift time colouring
    v_clean = v.replace(":", "")
    if "-" in v_clean:
        start_str = v_clean.split("-")[0].strip()
        if start_str.isdigit():
            t = int(start_str)
            if 500 <= t < 1200:
                return "background-color:#9BC2E6;color:black;"   # Morning
            if 1200 <= t < 1800:
                return "background-color:#FFE699;color:black;"   # Afternoon
            return "background-color:#CCC0DA;color:black;"       # Night
    return ""



In [13]:
def load_hc_master(folder_paths):
    hc_master_path = folder_paths.get("hc_master")
    if not hc_master_path:
        return None

    BASE_COLS = [
        "People ID", "IEX ID", "OracleID", "Employee Name", "Designation",
        "LOB", "TL ID", "Supervisor Name", "Email Id", "Wave", "CCT Training", "Grade"
    ]

    FINAL_COLS = [
        "People ID", "IEX ID", "OracleID", "Employee Name", "Designation",
        "LOB", "TL ID", "Supervisor Name", "Email Id", "Wave",
        "CCT Training", "Status", "JoinedDate", "LWD/Movement", "Grade"
    ]

    def ensure_cols(df: pl.DataFrame, cols: list[str]) -> pl.DataFrame:
        for col in cols:
            if col not in df.columns:
                df = df.with_columns(pl.lit(None).cast(pl.String).alias(col))
        return df

    def cast_to_string(df: pl.DataFrame, cols: list[str]) -> pl.DataFrame:
        return df.select([pl.col(c).cast(pl.String) for c in cols])

    # ── Active ──────────────────────────────────────────────────────────────
    df_active = pl.read_excel(hc_master_path, sheet_name="Active")
    df_active = ensure_cols(df_active, BASE_COLS)
    df_active = cast_to_string(df_active, BASE_COLS)
    df_active = df_active.with_columns([
        pl.lit("Active").alias("Status"),
        pl.col("CCT Training").alias("JoinedDate"),
        pl.lit(None).cast(pl.String).alias("LWD/Movement"),
    ])

    # ── Inactive ─────────────────────────────────────────────────────────────
    INACTIVE_COLS = BASE_COLS + ["LWD/Movement"]
    df_inactive = pl.read_excel(hc_master_path, sheet_name="Inactive")
    df_inactive = ensure_cols(df_inactive, INACTIVE_COLS)
    df_inactive = cast_to_string(df_inactive, INACTIVE_COLS)
    df_inactive = df_inactive.with_columns([
        pl.lit("Inactive").alias("Status"),
        pl.col("CCT Training").alias("JoinedDate"),
    ])

    # ── Merge & clean ────────────────────────────────────────────────────────
    hc_master_final = (
        pl.concat([df_active, df_inactive], how="diagonal")
        .with_columns(pl.col("Email Id").str.strip_chars())
        .filter(
            pl.col("Email Id").is_not_null() &
            (pl.col("Email Id") != "")
        )
        .unique(subset=["Email Id"], keep="first")
        .select(FINAL_COLS)
    )

    return hc_master_final
hc_master_df = load_hc_master(FOLDER_PATHS)

## HELPER: merge_schedules_horizontally() ────────────────────


In [14]:
# Scan a directory for all Schedule_WB*.xlsx files, find the relevant sheet
# (WB* or Hierarchy), auto-detect headers, unify date columns, infer
# Termination / Active status, and write a colour-coded Master Excel.

def merge_schedules_horizontally(
    base_folder: str,
    output_file: str = "Master_Schedule_Merged.xlsx",
) -> pd.DataFrame | None:
    """
    Merge every schedule workbook found under base_folder into one master file.

    Key logic:
    - Auto-detect header row (searches first 20 rows for IEX ID / EMP ID / ORACLE)
    - Rename columns for consistency (Email Id → Email, Emp ID → OracleID, etc.)
    - Use combine_first to merge overlapping date columns across files
    - Infer Termination by: explicit 'TERMINATION' value OR trailing-blank gap ≥ threshold
    - Fill remaining NaN: 'OFF' for agents, '' for managers (IEX ID is null)

    Edge cases:
    - File without a WB or Hierarchy sheet → skipped
    - Header not found in first 20 rows    → skipped
    - Non-date column that can't be parsed → silently ignored
    - combine_first: later files overwrite only NaN cells (first file wins)
    """
    TERMINATION_GAP = 7   # consecutive blank trailing days → mark Termination
    FIXED_COLS      = ["IEX ID", "OracleID", "Email", "Employee Name"]

    base_path = Path(base_folder)
    file_list = [f for f in base_path.rglob("*Schedule_WB*.xlsx")
                 if not f.name.startswith("~$")]

    print(f"Found {len(file_list)} Schedule_WB files.\n" + "-" * 60)

    master_df: pd.DataFrame | None = None

    for file in file_list:
        try:
            with pd.ExcelFile(_safe_read_excel.__wrapped__(file) if False else file) as xls:
                # Locate the primary sheet (WB* takes priority)
                wb_sheet = next((s for s in xls.sheet_names if s.upper().startswith("WB")), None)
                if not wb_sheet:
                    wb_sheet = next(
                        (s for s in ["Schedule_FINAL", "Schedule_VN"] if s in xls.sheet_names), None
                    )
                sheets = [s for s in [wb_sheet, "Hierarchy"] if s and s in xls.sheet_names]
                if not sheets:
                    print(f"  [SKIP] {file.name}: no usable sheet found.")
                    continue

                for sheet in sheets:
                    raw = pd.read_excel(xls, sheet_name=sheet, header=None)

                    # Auto-detect header row
                    header_row = -1
                    for idx, row in raw.head(20).iterrows():
                        tokens = [str(x).strip().upper() for x in row if pd.notna(x)]
                        if any(k in t for t in tokens for k in ["IEX ID", "EMP ID", "ORACLE"]):
                            header_row = idx
                            break
                    if header_row == -1:
                        print(f"  [SKIP] {file.name} / {sheet}: header not found.")
                        continue

                    df = pd.read_excel(xls, sheet_name=sheet, header=header_row)
                    df.rename(columns=lambda x: str(x).strip(), inplace=True)

                    # Column name normalisation
                    rename_map: dict = {}
                    if sheet == "Hierarchy":
                        rename_map.update({"Emp ID": "OracleID", "Name": "Employee Name"})
                        df["IEX ID"] = pd.NA
                        df["Email"]  = ""
                    else:
                        rename_map.update({
                            "Email Id": "Email", "Email ID": "Email", "Name": "Employee Name",
                        })
                    df.rename(columns=rename_map, inplace=True)

                    df["OracleID"] = pd.to_numeric(df.get("OracleID"), errors="coerce")
                    df = df.dropna(subset=["OracleID"]).copy()
                    if df.empty:
                        continue

                    # Identify and rename date columns
                    date_rename: dict = {}
                    keep = [c for c in FIXED_COLS if c in df.columns]
                    for col in df.columns:
                        if col in FIXED_COLS:
                            continue
                        try:
                            dt = pd.to_datetime(col)
                            date_rename[col] = dt.strftime("%Y-%m-%d")
                            keep.append(col)
                        except Exception:
                            pass

                    df = df[keep].rename(columns=date_rename)

                    if sheet == "Hierarchy":
                        date_c = list(date_rename.values())
                        df[date_c] = df[date_c].replace("WO", "OFF")
                        df[date_c] = df[date_c].replace(["0", "nan", "", np.nan], np.nan)

                    df = df.set_index("OracleID")
                    master_df = df if master_df is None else master_df.combine_first(df)

            print(f"  [+] Merged: {file.name}")

        except Exception as exc:
            print(f"  [!] Error with {file.name}: {exc}")

    if master_df is None or master_df.empty:
        print("No data to merge.")
        return None

    master_df = master_df.reset_index()
    master_df["IEX ID"]   = pd.to_numeric(master_df.get("IEX ID"),   errors="coerce")
    master_df["OracleID"] = pd.to_numeric(master_df.get("OracleID"), errors="coerce")

    # Sort: agents first (have IEX ID), then managers
    master_df["_is_mgr_"] = master_df["IEX ID"].isna()
    master_df = master_df.sort_values(["_is_mgr_", "OracleID"]).drop(columns=["_is_mgr_"])

    # Re-order: fixed cols then sorted date cols
    all_date_cols = sorted(
        [c for c in master_df.columns if c not in FIXED_COLS and c != "Status"],
        key=lambda x: pd.to_datetime(x, errors="coerce") or pd.Timestamp.min,
    )
    master_df = master_df[[c for c in FIXED_COLS if c in master_df.columns] + all_date_cols]

    # ── Termination inference ─────────────────────────────────────────────────
    df_d = master_df[all_date_cols].copy()
    df_d = df_d.replace(r"^\s*$", np.nan, regex=True).replace(["nan", "EMPTY_H"], np.nan)

    term_mask = df_d.apply(
        lambda s: s.astype(str).str.contains("TERMINATION", case=False, na=False)
    )
    df_d[term_mask] = "Termination"

    # Forward-fill Termination across subsequent blank cells
    ffilled = df_d.ffill(axis=1)
    df_d[(ffilled == "Termination") & df_d.isna()] = "Termination"

    # Infer Termination from trailing blanks (≥ threshold)
    col_idx = np.arange(len(all_date_cols))
    is_valid = df_d.notna() & (df_d != "Termination")

    last_valid = np.where(is_valid, col_idx, -1).max(axis=1)
    trailing   = len(all_date_cols) - 1 - last_valid
    rows_update = (trailing >= TERMINATION_GAP) & (last_valid >= 0)

    col_2d = np.tile(col_idx, (len(df_d), 1))
    df_d.values[(col_2d > last_valid[:, None]) & rows_update[:, None]] = "Termination"

    # Clear leading blanks (before first valid shift)
    first_valid = np.where(is_valid, col_idx, len(all_date_cols)).min(axis=1)
    df_d.values[col_2d < first_valid[:, None]] = ""

    # Derive Status
    is_inactive = (df_d == "Termination").any(axis=1)
    master_df.insert(
        master_df.columns.get_loc("Employee Name") + 1,
        "Status",
        np.where(is_inactive, "Inactive", "Active"),
    )
    master_df[all_date_cols] = df_d

    # Fill remaining NaN
    is_mgr = master_df["IEX ID"].isna()
    master_df.loc[~is_mgr, all_date_cols] = master_df.loc[~is_mgr, all_date_cols].fillna("OFF")
    master_df.loc[is_mgr,  all_date_cols] = master_df.loc[is_mgr,  all_date_cols].fillna("")

    master_df = master_df.dropna(subset=["OracleID", "Employee Name"])

    if 'hc_master_df' in globals() and hc_master_df is not None:
        # Convert Polars → pandas if needed
        _hc = hc_master_df.to_pandas() if hasattr(hc_master_df, 'to_pandas') else hc_master_df.copy()
        _hc['OracleID'] = pd.to_numeric(_hc['OracleID'], errors='coerce')

        # Build OracleID → Email lookup (drop rows with null email)
        _email_map = (
            _hc[['OracleID', 'Email Id']]
            .dropna(subset=['OracleID', 'Email Id'])
            .drop_duplicates(subset=['OracleID'], keep='first')
            .set_index('OracleID')['Email Id']
            .str.strip()
        )

        if 'Email' not in master_df.columns:
            master_df.insert(master_df.columns.get_loc('Employee Name'), 'Email', '')

        # Fill only blank / NaN Email cells — never overwrite existing values
        _blank_mask = master_df['Email'].isna() | (master_df['Email'].astype(str).str.strip() == '')
        master_df.loc[_blank_mask, 'Email'] = (
            master_df.loc[_blank_mask, 'OracleID'].map(_email_map)
        )

        _filled = _blank_mask.sum() - (master_df['Email'].isna() | (master_df['Email'].astype(str).str.strip() == '')).sum()
        print(f'    [INFO] Email filled: {_filled} cells updated from hc_master_df.')
    else:
        print('    [WARN] hc_master_df not found — Email column left as-is.')

    # Apply colour styling and export
    styler = master_df.style
    apply_fn = styler.map if hasattr(styler, "map") else styler.applymap
    styled = apply_fn(get_cell_style, subset=all_date_cols)

    output_path = base_path / output_file
    styled.to_excel(output_path, index=False, engine="openpyxl")
    print(f"\n[✅ COMPLETED] Master file saved: {output_path}")
    return master_df


# Example call
base_dir = f"{HOME}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Schedule/Schedule (Ops version)/2026"
df_master = merge_schedules_horizontally(base_dir)

Found 20 Schedule_WB files.
------------------------------------------------------------
  [+] Merged: Expedia VN_Schedule_WB0105.xlsx
  [+] Merged: Expedia VN_Schedule_WB0112.xlsx
  [+] Merged: Expedia VN_Schedule_WB0119.xlsx
  [+] Merged: Expedia VN_Schedule_WB0126.xlsx
  [+] Merged: Expedia VN_Schedule_WB0202.xlsx
  [+] Merged: Expedia VN_Schedule_WB0209.xlsx
  [+] Merged: Expedia VN_Schedule_WB0216.xlsx
  [+] Merged: Expedia VN_Schedule_WB0223.xlsx
  [+] Merged: Expedia VN_Schedule_WB0302.xlsx
  [+] Merged: Expedia VN_Schedule_WB0309.xlsx
  [+] Merged: Expedia VN_Schedule_WB0316.xlsx
  [+] Merged: Expedia VN_Schedule_WB0323.xlsx
  [+] Merged: Expedia VN_Schedule_WB0330.xlsx
  [+] Merged: Expedia VN_Schedule_WB0406.xlsx
  [+] Merged: Expedia VN_Schedule_WB0413.xlsx
  [+] Merged: Expedia VN_Schedule_WB0420.xlsx
  [+] Merged: Expedia VN_Schedule_WB0427.xlsx
  [+] Merged: Expedia VN_Schedule_WB0504.xlsx
  [+] Merged: Expedia VN_Schedule_WB0511.xlsx
  [+] Merged: Expedia VN_Schedule_WB0

## DEBUG TOOL: track_agent_schedule() ────────────────────────


In [15]:
# Produces a horizontal step-by-step tracking table for a single agent/week.
# Shows: Raw → Restore → Swap instructions & results → Planned Leave → Final.

# Ensure _normalise_name is available even if Cell 18 hasn't run yet
import re as _re
if '_normalise_name' not in globals():
    def _normalise_name(raw) -> str:
        s = str(raw).strip()
        if not s or s.lower() in ('nan', 'none', ''): return ''
        if ',' in s:
            parts = [p.strip() for p in s.split(',', 1)]
            first = _re.sub(r'(?<=[a-z])(?=[A-Z])', ' ', parts[1])
            s = first + ' ' + parts[0]
        return s.upper().strip()

def track_agent_schedule(target_iex: int, target_week: str) -> pd.DataFrame:
    """
    Trace every transformation applied to an agent's schedule in a readable table.

    Columns: Stage | Mon_date | Tue_date | ... | Notes
    Rows:    Original | Restore | Swap step instructions & results | Planned | Final

    Edge cases:
    - Agent not in raw schedule     → blank row with note
    - Agent has no swaps            → 'No swap' row
    - Partner not in raw schedule   → partner shift shown as 'UNKNOWN'
    - Swap with leave-restored value → correctly simulates partner's restored shift
    """
    INFO_COLS = {"Week", "IEX ID", "Name", "TL", "Status", "Sec Status", "LOB",
                 "Detailed Swap History", "Swap Steps", "Planned Note", "Planned"}

    # Guard: verify all required DataFrames exist (run Cells 7-10 first)
    required = {
        "Schedule_VN":      "Cell 9  — main pipeline",
        "schedule_step_1":  "Cell 9  — main pipeline",
        "Schedule_Swap":    "Cell 9  — main pipeline",
        "Schedule_Planned": "Cell 9  — main pipeline",
    }
    missing = [f"{var} (defined in {cell})" for var, cell in required.items() if var not in globals()]
    if missing:
        print("[ERROR] The following variables are not defined — please run the indicated cells first:")
        for m in missing:
            print(f"  ❌  {m}")
        return pd.DataFrame()


    date_cols_7 = [c for c in Schedule_VN.columns if c not in INFO_COLS][:7]
    current = {d: "" for d in date_cols_7}
    rows    = []

    # ── 1. Original (raw) ────────────────────────────────────────────────────
    r_raw = {"Stage": "★ ORIGINAL (Raw)"}
    if "schedule_step_1" in globals():
        hit = schedule_step_1[
            (schedule_step_1["Week"] == target_week) & (schedule_step_1["IEX ID"] == target_iex)
        ]
        if not hit.empty:
            for d in date_cols_7:
                v = str(hit.iloc[0].get(d, ""))
                r_raw[d], current[d] = v, v
            r_raw["Notes"] = "Initial unprocessed schedule"
        else:
            r_raw["Notes"] = "Agent not found in raw schedule"
    rows.append(r_raw)

    # ── 1b. Restore base shift (if any global leave codes present) ───────────
    LEAVE_RESTORE = {"PTO", "PAID LEAVE", "HOLIDAY", "AL", "CO", "HO", "LWP", "LEAVE"}
    if any(str(v).strip().upper() in LEAVE_RESTORE for v in current.values()):
        r_res = {"Stage": "   ↳ Restore Base Shift"}
        shifts_only = [v for v in current.values() if "-" in str(v)]
        mode_res    = pd.Series(shifts_only).mode()
        dominant    = mode_res.iloc[0] if not mode_res.empty else "0000-0000"
        notes_r     = []
        for d in date_cols_7:
            if str(current[d]).strip().upper() in LEAVE_RESTORE:
                notes_r.append(f"{d[-5:]}: {current[d]} → {dominant}")
                current[d] = dominant
            r_res[d] = current[d]
        r_res["Notes"] = "Restored: " + " | ".join(notes_r)
        rows.append(r_res)

    # ── 2. Swap steps ─────────────────────────────────────────────────────────
    if "Schedule_Swap" in globals():
        agent_swaps = Schedule_Swap[
            (Schedule_Swap["IEX1"] == target_iex) | (Schedule_Swap["IEX2"] == target_iex)
        ]
        if agent_swaps.empty:
            rows.append({"Stage": "➔ Swap Info", "Notes": "No swap requested"})
        else:
            step_ctr: dict = {}
            for _, r in agent_swaps.iterrows():
                iex1, iex2 = r["IEX1"], r["IEX2"]
                step_val = str(r.get("Step", "?")).strip()
                step_ctr[step_val] = step_ctr.get(step_val, -1) + 1
                cnt   = step_ctr[step_val]
                label = f"Swap {step_val}" + (f".{cnt}" if cnt else "")

                is_self = pd.isna(iex2) or iex1 == iex2
                partner = iex2 if iex1 == target_iex else iex1
                # Resolve partner name: target is iex1 → partner name is Name2, else Name1
                partner_name = str(r.get('Name2','') if iex1 == target_iex else r.get('Name1','')).strip()
                partner_name = _normalise_name(partner_name) if partner_name else ''

                # Instruction row
                r_inst = {"Stage": f"➔ {label} (Instruction)"}
                for i, day in enumerate(WEEKDAYS):
                    if i < len(date_cols_7):
                        v = str(r.get(day, "")).strip()
                        r_inst[date_cols_7[i]] = "" if v.upper() in ("NAN", "NONE") else v
                r_inst["Notes"] = (
                    "Self-change"
                    if is_self else
                    f"Swapped with {partner_name} (IEX: {int(partner) if pd.notna(partner) else partner})"
                )
                rows.append(r_inst)

                # Result row
                r_res2 = {"Stage": f"   ↳ Result after {label}"}
                notes2 = []
                for i, day in enumerate(WEEKDAYS):
                    if i >= len(date_cols_7):
                        continue
                    dc  = date_cols_7[i]
                    val = str(r.get(day, "")).strip()
                    vup = val.upper()
                    if is_self:
                        if vup not in ("KEEP", "SWAP", "NAN", "NONE", ""):
                            current[dc] = val
                            notes2.append(f"{day[:3]}: {val}")
                    elif vup == "SWAP":
                        p_shift = "UNKNOWN"
                        if "schedule_step_1" in globals():
                            p_hit = schedule_step_1[
                                (schedule_step_1["Week"] == target_week)
                                & (schedule_step_1["IEX ID"] == partner)
                            ]
                            if not p_hit.empty:
                                p_raw  = p_hit.iloc[0].to_dict()
                                p_vals = [str(p_raw.get(d2, "")).strip()
                                          for d2 in date_cols_7 if "-" in str(p_raw.get(d2, ""))]
                                p_mode = pd.Series(p_vals).mode()
                                p_dom  = p_mode.iloc[0] if not p_mode.empty else "0000-0000"
                                cell_v = str(p_raw.get(dc, "")).strip().upper()
                                p_shift = p_dom if cell_v in LEAVE_RESTORE else str(p_raw.get(dc, ""))
                        current[dc] = p_shift
                        notes2.append(f"{day[:3]}: got {p_shift}")
                    r_res2[dc] = current[dc]
                r_res2["Notes"] = ", ".join(notes2) if notes2 else "No changes"
                rows.append(r_res2)

    # ── 3. Planned Leave ──────────────────────────────────────────────────────
    r_lv = {"Stage": "➔ Plot Leave (Planned)"}
    r_lv_res = {"Stage": "   ↳ Result after Planned"}
    has_leave = False

    if "Schedule_Planned" in globals():
        agent_leave = Schedule_Planned[Schedule_Planned["IEX ID"] == target_iex]
        notes_lv = []
        for d in date_cols_7:
            vals = agent_leave[agent_leave["Leave Date"].astype(str) == str(d)]["Type of Request"].tolist()
            if vals:
                has_leave = True
                req = " | ".join(str(v) for v in vals)
                r_lv[d] = req
                if str(current[d]).upper() != "OFF":
                    current[d] = req
                    notes_lv.append(f"{d[-5:]}: Applied {req}")
                else:
                    notes_lv.append(f"{d[-5:]}: Ignored {req} (OFF)")
            else:
                r_lv[d] = ""
            r_lv_res[d] = current[d]
        r_lv["Notes"] = "Leave applied" if has_leave else "No leave this week"
        r_lv_res["Notes"] = " | ".join(notes_lv)

    rows.append(r_lv)
    if has_leave:
        rows.append(r_lv_res)

    # ── 4. Final ──────────────────────────────────────────────────────────────
    r_fin = {"Stage": "★ FINAL (Schedule_VN)"}
    fin_hit = Schedule_VN[
        (Schedule_VN["Week"] == target_week) & (Schedule_VN["IEX ID"] == target_iex)
    ]
    if not fin_hit.empty:
        for d in date_cols_7:
            r_fin[d] = str(fin_hit.iloc[0].get(d, ""))
        hist = str(fin_hit.iloc[0].get("Detailed Swap History", "")).replace("\n", " ")
        note = str(fin_hit.iloc[0].get("Planned Note", ""))
        parts = []
        if hist.strip(): parts.append(f"[History: {hist}]")
        if note.strip(): parts.append(f"[Leave Note: {note}]")
        r_fin["Notes"] = " ".join(parts) if parts else ""
    else:
        r_fin["Notes"] = "Agent not found in Schedule_VN"
    rows.append(r_fin)

    # ── Render ────────────────────────────────────────────────────────────────
    tracking_df = pd.DataFrame(rows).fillna("")
    print(f"🔍 TRACKING TABLE — IEX: {target_iex} | WEEK: {target_week}")
    print("=" * 110)
    with pd.option_context("display.max_rows", None, "display.max_columns", None,
                           "display.max_colwidth", None):
        display(HTML(tracking_df.to_html(index=False)))

    return tracking_df


# ── Example usage ─────────────────────────────────────────────────────────────
TARGET_IEX_1 = 3085545
TARGET_IEX_2 = 3096698

df_track_1 = track_agent_schedule(TARGET_IEX_1, WEEK)
df_track_2 = track_agent_schedule(TARGET_IEX_2, WEEK)

🔍 TRACKING TABLE — IEX: 3085545 | WEEK: Schedule_2026_05_18


Stage,2026-05-18,2026-05-19,2026-05-20,2026-05-21,2026-05-22,2026-05-23,2026-05-24,Notes
★ ORIGINAL (Raw),2000-0500,OFF,OFF,2000-0500,2000-0500,2000-0500,2000-0500,Initial unprocessed schedule
➔ Swap Info,,,,,,,,No swap requested
➔ Plot Leave (Planned),,,,,,,,No leave this week
★ FINAL (Schedule_VN),2000-0500,OFF,OFF,2000-0500,2000-0500,2000-0500,2000-0500,


🔍 TRACKING TABLE — IEX: 3096698 | WEEK: Schedule_2026_05_18


Stage,2026-05-18,2026-05-19,2026-05-20,2026-05-21,2026-05-22,2026-05-23,2026-05-24,Notes
★ ORIGINAL (Raw),0600-1500,OFF,OFF,0600-1500,0600-1500,0600-1500,0600-1500,Initial unprocessed schedule
➔ Swap 2 (Instruction),SWAP,SWAP,SWAP,SWAP,SWAP,SWAP,SWAP,Swapped with HO QUY MINH (IEX: 3026445)
↳ Result after Swap 2,0700-1600,OFF,OFF,0700-1600,0700-1600,0700-1600,0700-1600,"Mon: got 0700-1600, Tue: got OFF, Wed: got OFF, Thu: got 0700-1600, Fri: got 0700-1600, Sat: got 0700-1600, Sun: got 0700-1600"
➔ Plot Leave (Planned),,,,,,,,No leave this week
★ FINAL (Schedule_VN),0700-1600,OFF,OFF,0700-1600,0700-1600,0700-1600,0700-1600,[History: 3096698 - NGUYEN NGOC DIEM QUYNH <-> 3026445 - HO QUY MINH (Step 2)]


## DEBUG: QUICK AGENT LOOKUP ──────────────────────────────────


In [16]:
# NOTE (BUG FIX): Cells 9/10/11 in the original used 'schedule_step_1' as filter
# base but displayed Schedule_VN — these should both reference the same DataFrame.
# The correct query is shown below.

# Change these two values to inspect any agent/week combination
DEBUG_IEX  = 3085545
DEBUG_WEEK = WEEK   # or e.g. "Schedule_2025_06_09"

Schedule_VN[
    (Schedule_VN["Week"] == DEBUG_WEEK) & (Schedule_VN["IEX ID"] == DEBUG_IEX)
]


,Week,IEX ID,Name,TL,Status,Sec Status,LOB,2026-05-18,2026-05-19,2026-05-20,2026-05-21,2026-05-22,2026-05-23,2026-05-24,Planned Note,Detailed Swap History,Swap Steps,Planned
1,Schedule_2026_05_18,3085545,"DOAN, ANNY","Ann,THAO",Active,Production,Lodging,2000-0500,OFF,OFF,2000-0500,2000-0500,2000-0500,2000-0500,,,,0


In [17]:
# ramco_file = r"C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Save\EXP - CO Tracking.xlsx"
# vn_team_file = FOLDER_PATHS['vn_team']  
# schedule_team_file = os.path.join(FOLDER_PATHS['schedule_team'], f"{WEEK}.xlsx")
# output_csv_path = "EXP_CO_Tracking_Export.csv"

# df_ramco = pd.read_excel(ramco_file, sheet_name="ramco_rawdata")
# df_ramco['Date'] = pd.to_datetime(df_ramco['Date'], errors='coerce')
# df_ramco['ramco_marked'] = df_ramco['ramco_marked'].astype(str).str.strip().str.upper()

# df_hc = pd.read_excel(vn_team_file, sheet_name="HC", header=None)
# df_hc.columns = df_hc.iloc[0]
# df_hc = df_hc[1:][['OracleID', 'IEX ID', 'Employee Name']].copy()
# df_hc['OracleID'] = pd.to_numeric(df_hc['OracleID'], errors='coerce')
# df_hc['IEX ID'] = pd.to_numeric(df_hc['IEX ID'], errors='coerce')
# df_hc = df_hc.dropna(subset=['OracleID', 'IEX ID']).drop_duplicates(subset=['OracleID'])

# df_ramco['EID'] = pd.to_numeric(df_ramco['EID'], errors='coerce')
# df_merged = pd.merge(df_ramco, df_hc, left_on='EID', right_on='OracleID', how='inner')

# holidays_data = [
#     ("2023-01-01", "New Year's Day"), ("2023-01-21", "TET"), ("2023-01-22", "TET"), ("2023-01-23", "TET"), ("2023-01-24", "TET"), ("2023-01-25", "TET"),
#     ("2023-04-29", "KING'S DAY"), ("2023-04-30", "Reunification Day"), ("2023-05-01", "Labor Day"), ("2023-09-01", "Independence Day"), ("2023-09-02", "Independence Day"),
#     ("2024-01-01", "New Year's Day"), ("2024-02-09", "TET"), ("2024-02-10", "TET"), ("2024-02-11", "TET"), ("2024-02-12", "TET"), ("2024-02-13", "TET"),
#     ("2024-04-18", "KING'S DAY"), ("2024-04-30", "Reunification Day"), ("2024-05-01", "Labor Day"), ("2024-09-02", "Independence Day"), ("2024-09-03", "Independence Day"),
#     ("2025-01-01", "New Year's Day"), ("2025-01-28", "TET"), ("2025-01-29", "TET"), ("2025-01-30", "TET"), ("2025-01-31", "TET"), ("2025-02-01", "TET"),
#     ("2025-04-07", "KING'S DAY"), ("2025-04-30", "Reunification Day"), ("2025-05-01", "Labor Day"), ("2025-09-01", "Independence Day"), ("2025-09-02", "Independence Day")
# ]
# df_holidays = pd.DataFrame(holidays_data, columns=['Date', 'Holiday_Name'])
# df_holidays['Date'] = pd.to_datetime(df_holidays['Date'])
# df_holidays['Is_Holiday'] = 1

# df = pd.merge(df_merged, df_holidays, on='Date', how='left')
# df['Is_Holiday'] = df['Is_Holiday'].fillna(0)

# df['Is_WO'] = np.where(df['ramco_marked'] == 'WO', 1, 0)
# df['Is_PH'] = np.where(df['ramco_marked'] == 'PH', 1, 0)
# df['Is_PO'] = np.where(df['ramco_marked'] == 'PO', 1, 0)
# df['Is_NM'] = np.where(df['ramco_marked'] == 'NM', 1, 0)
# df['WO_on_Holiday'] = np.where((df['Is_Holiday'] == 1) & (df['Is_WO'] == 1), 1, 0)

# df['Week_Start_Monday'] = df['Date'] - pd.to_timedelta(df['Date'].dt.weekday, unit='d')
# df['Week_Start_Monday'] = df['Week_Start_Monday'].dt.strftime('%Y-%m-%d')

# df['Holiday_Info'] = np.where(df['Is_Holiday'] == 1, df['Date'].dt.strftime('%d/%m') + " - " + df['Holiday_Name'], "")
# df['Daily_Log'] = df['Date'].dt.strftime('%d/%m') + " - " + df['ramco_marked']

# weekly_summary = df.groupby(['IEX ID', 'OracleID', 'Employee Name', 'Week_Start_Monday']).agg(
#     Total_OFF=('Is_WO', 'sum'),
#     Total_PH=('Is_PH', 'sum'),
#     Total_PO=('Is_PO', 'sum'),
#     Total_NM=('Is_NM', 'sum'),
#     Target_HO=('Is_Holiday', 'sum'),
#     WO_on_Holiday=('WO_on_Holiday', 'sum'),
#     Holiday_Notes=('Holiday_Info', lambda x: " | ".join(sorted(list(set(i for i in x if i != ""))))),
#     Log=('Daily_Log', lambda x: ", ".join([i for i in x if i != ""]))
# ).reset_index()

# weekly_summary['Total_CO_Taken'] = np.maximum(0, weekly_summary['Total_OFF'] - 2)
# weekly_summary['Applied_HO'] = np.maximum(0, weekly_summary['Target_HO'] - weekly_summary['Total_PH'])
# missing_wo = np.maximum(0, 2 - (weekly_summary['Total_OFF'] + weekly_summary['Total_PO']))
# weekly_summary['Total_CO_Earned'] = np.minimum(weekly_summary['Applied_HO'], missing_wo + weekly_summary['WO_on_Holiday'])

# final_columns = ['IEX ID', 'OracleID', 'Employee Name', 'Week_Start_Monday', 'Total_OFF', 'Total_PH', 'Total_PO', 'Total_NM', 'Target_HO', 'Applied_HO', 'Total_CO_Earned', 'Total_CO_Taken', 'Holiday_Notes', 'Log']
# weekly_summary = weekly_summary[final_columns].sort_values(by=['IEX ID', 'Week_Start_Monday'])
# weekly_summary = weekly_summary[(weekly_summary['Total_OFF'] > 0) | (weekly_summary['Target_HO'] > 0) | (weekly_summary['Total_PH'] > 0) | (weekly_summary['Total_PO'] > 0)]
# weekly_summary.to_csv(output_csv_path, index=False, encoding='utf-8-sig')